## Performance prediction (per metric, repetitions)

Runs the particle filter **per performance metric** on the test units and saves the
per-step predicted RUL band to one CSV per repetition, mirroring `6-rul_test_rep.ipynb`.
The plotting notebook only **reads** these files. Baseline args (`NETWORK=None`) load
the per-metric Optuna gains; network args load each metric's trained controller.

In [23]:
import os

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed

from experiment_config import (
    SEED,
    UNCERTAINTY_LEVEL,
    DegModel,
    dataset_paths,
    load_baseline_gains,
    pfnet_paths,
)
from src.helpers.seed import set_global_seed
from src.models.networks.pf_mlp import ParticleFilterMLP, build_activation
from src.models.particle_filter.core import ParticleFilter
from src.training.pfnet_hparams import PFNET_ARGS

## Configuration

In [24]:
DATA_NAME = "DS05"
ARGS_ID = 7
N_REP = 10

perform_name = "SmLPC"


## Parameters

In [25]:
set_global_seed(SEED)
ESTIMATION_DIR, DEGR_MODEL_DIR = dataset_paths(
    DATA_NAME, fields=["estimation", "degr_model"]
)

# CPUs allocated by SLURM (fallback 4); repetitions run in parallel across these
N_CPUS = int(os.environ.get("SLURM_CPUS_PER_TASK") or 10)


args = PFNET_ARGS[ARGS_ID]
net_args = args["NETWORK"]
USE_NET = net_args is not None  # net_args None -> no-network PF baseline
pf_args = args["PARTICLE_FILTER"]
MAX_LIFE = int(pf_args["MAX_LIFE"])

# per-metric folder, mirroring training/optuna layout net_arg{ARGS_ID}/{perform_name}
PFNET_ARGS_DIR, _ = pfnet_paths(ARGS_ID, DATA_NAME)
PERFORM_DIR = PFNET_ARGS_DIR / perform_name
PERFORM_DIR.mkdir(parents=True, exist_ok=True)
print("N_CPUS:", N_CPUS)


N_CPUS: 10


## Load base models and test data

In [26]:
# dev units -> shared base degradation-model population (same as 6-rul)
dev_units = (
    pd.read_csv(ESTIMATION_DIR / "data_dev.csv")["unit"].astype(int).unique().tolist()
)

test_hi = pd.read_csv(ESTIMATION_DIR / "data_test.csv")
test_units = test_hi["unit"].astype(int).unique().tolist()
performs = {u: test_hi[test_hi["unit"] == u][perform_name].values for u in test_units}
time = {u: test_hi[test_hi["unit"] == u]["cycle"].values for u in test_units}
print("test units:", test_units, "| metric:", perform_name)


test units: [7, 8, 9, 10] | metric: SmLPC


## Build per-metric filters

In [ ]:
# controller (network) or tuned gains (baseline) + base models for this metric
if USE_NET:
    net = ParticleFilterMLP(
        state_dim=DegModel.state_dim(),
        hidden_dims=list(net_args["HIDDEN_DIMS"]),
        activation=build_activation(net_args),
        dropout_p=float(net_args["DROPOUT"]),
    )
    ckpt = torch.load(
        PFNET_ARGS_DIR / perform_name / "checkpoint_best.pt", weights_only=False
    )
    net.load_state_dict(ckpt["model_state"])
    net = net.eval()
    metric_gains = {}
else:
    net = None
    metric_gains = load_baseline_gains(ARGS_ID, DATA_NAME, perform_name)

degmodels = []
for u in dev_units:
    m = DegModel()
    m.load_state_dict(
        torch.load(
            DEGR_MODEL_DIR / "states" / perform_name / f"unit_{u}" / "best_model.pt"
        )
    )
    degmodels.append(m)


def make_pf():
    return ParticleFilter(
        base_models=degmodels,
        net=net,
        n_particles=int(pf_args["N_PARTICLES"]),
        max_life=MAX_LIFE,
        use_net=USE_NET,
        const_noise=metric_gains.get("NOISE"),
        const_prior=metric_gains.get("PRIOR"),
        const_lik=metric_gains.get("LIK"),
    ).eval()


## Predict RUL per metric and save

In [ ]:
# Each repetition is independent (own seed + own CSV), so run them in parallel
# across CPUs. Existing rep files are skipped (crash-safe / resumable). Columns
# match plot_rul_from_dataframe: time, mean, lower, upper, true_rul.
n_workers = max(1, min(N_REP, N_CPUS))
threads_per_worker = max(1, N_CPUS // n_workers)


def run_rep(rep):
    rep_path = PERFORM_DIR / f"perform_test_r{rep}.csv"
    if rep_path.exists():
        return f"rep {rep + 1}/{N_REP}: already saved ({rep_path.name})"

    torch.set_num_threads(threads_per_worker)
    set_global_seed(SEED + rep)

    rows = []
    for unit in test_units:
        t_np, s_np = time[unit], performs[unit]
        eol_time = float(t_np[-1])
        t_data = torch.tensor(t_np, dtype=torch.float32)
        s_data = torch.tensor(s_np, dtype=torch.float32)

        pf = make_pf()  # fresh filter per unit
        for k in range(len(t_data)):
            pf.step(t_obs=t_data[[k]], s_obs=s_data[[k]])
            lo, me, up = pf.mixture.uncertainty_interval(
                s=torch.zeros(1), level=UNCERTAINTY_LEVEL
            )
            tk = float(t_np[k])
            rows.append(
                {
                    "perform": perform_name,
                    "unit": unit,
                    "rep": rep,
                    "seed": SEED + rep,
                    "time": tk,
                    "s_obs": float(s_np[k]),
                    "mean": float(np.clip(me.item() - tk, 0.0, MAX_LIFE)),
                    "lower": float(np.clip(lo.item() - tk, 0.0, MAX_LIFE)),
                    "upper": float(np.clip(up.item() - tk, 0.0, MAX_LIFE)),
                    "true_rul": max(eol_time - tk, 0.0),
                }
            )

    pd.DataFrame(rows).to_csv(rep_path, index=False)
    return f"rep {rep + 1}/{N_REP}: saved {rep_path.name} ({len(rows)} rows)"


print(f"running {N_REP} reps on {n_workers} workers x {threads_per_worker} threads")
for msg in Parallel(n_jobs=n_workers, backend="loky", return_as="generator")(
    delayed(run_rep)(rep) for rep in range(N_REP)
):
    print(msg, flush=True)


running 10 reps on 10 workers x 1 threads


rep 1/10: saved perform_test_r0.csv (327 rows)
rep 2/10: saved perform_test_r1.csv (327 rows)
rep 3/10: saved perform_test_r2.csv (327 rows)
rep 4/10: saved perform_test_r3.csv (327 rows)
rep 5/10: saved perform_test_r4.csv (327 rows)
rep 6/10: saved perform_test_r5.csv (327 rows)
rep 7/10: saved perform_test_r6.csv (327 rows)
rep 8/10: saved perform_test_r7.csv (327 rows)
rep 9/10: saved perform_test_r8.csv (327 rows)
rep 10/10: saved perform_test_r9.csv (327 rows)
